# CAD-Gym — RL Training on Colab (Unsloth + LoRA)

Train a language model to design mechanical brackets and watch it improve in real-time.

**Memory strategy:** Unsloth 4-bit + LoRA + gradient checkpointing. Fits Qwen2.5-Coder-7B
on a T4 (16 GB VRAM) with room to spare. Faster and more stable than vanilla fine-tuning.

**Runtime:** GPU required. Set runtime → T4 GPU before running.

| Model | VRAM (4-bit LoRA) | Speed | Notes |
|-------|-------------------|-------|-------|
| Qwen2.5-Coder-1.5B | ~1 GB | fast | good for quick tests |
| Qwen2.5-Coder-7B | ~5 GB | medium | **recommended** |
| Qwen2.5-Coder-14B | ~9 GB | slower | needs A100 |


## 1. Install

In [ ]:
%%capture
# Unsloth: 4-bit quantised model loading + LoRA + fused kernels
# Must install before transformers to avoid version conflicts
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Prodata + CAD dependencies
!pip install cadquery trimesh gymnasium pydantic
!pip install trl datasets plotly tqdm

# Install prodata from GitHub (or use local: pip install -e .)
!pip install git+https://github.com/prodata-ai/ProdataGym.git

import plotly.io as pio
pio.renderers.default = 'colab'
print('Installation complete')

## 2. Load environment + model (Unsloth 4-bit + LoRA)

In [ ]:
import os
import gymnasium as gym
import prodata.cad_gym
from prodata.cad_gym.viz import TrainingVisualizer

import torch
import numpy as np
from tqdm.notebook import tqdm

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

env = gym.make('prodata/BracketDesign-v0')
print(f'Environment: {len(env.unwrapped.task_ids())} tasks loaded')

# ─── Model selection ─────────────────────────────────────────────────────────
# Unsloth hosts 4-bit pre-quantised models at unsloth/<model>-bnb-4bit
# These download faster and skip the quantisation step at startup.
MODEL_NAME = 'unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit'
# Alternatives:
#   'unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit'  # fast, weaker
#   'unsloth/Qwen2.5-Coder-14B-Instruct-bnb-4bit'   # better, needs A100

MAX_SEQ_LEN = 1024   # bracket code is short; 1024 is plenty
LORA_RANK   = 16     # r=16: good balance of capacity vs VRAM

print(f'Loading {MODEL_NAME} (4-bit)...')

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LEN,
    load_in_4bit    = True,      # 4-bit NF4 quantisation — halves VRAM vs fp16
    dtype           = None,      # auto-detect: bf16 on Ampere, fp16 on older
)

# Add LoRA adapters — only these weights get gradient updates
# For a 7B model: all params = 7B, trainable LoRA params ≈ 40M (~0.6%)
model = FastLanguageModel.get_peft_model(
    model,
    r                = LORA_RANK,
    target_modules   = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                        'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha       = LORA_RANK,   # alpha = r → effective LR scale = 1
    lora_dropout     = 0,           # 0 = no dropout (Unsloth recommendation)
    bias             = 'none',
    use_gradient_checkpointing = 'unsloth',  # Unsloth's patched version: saves
                                             # 30% more VRAM than HF checkpointing
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = next(model.parameters()).device

# Optimizer: standard AdamW is fine now — only LoRA params (~40M) need state,
# not the full 7B. Optimizer memory is ~300 MB instead of ~28 GB.
from torch.optim import AdamW
optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr     = 2e-4,   # higher LR works for LoRA (smaller parameter space)
    weight_decay = 0.01,
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total_params/1e6:.0f}M')
print(f'Trainable (LoRA): {trainable_params/1e6:.1f}M ({trainable_params/total_params:.2%})')
if torch.cuda.is_available():
    print(f'VRAM used:        {torch.cuda.memory_allocated()/1e9:.1f} GB')
    print(f'VRAM reserved:    {torch.cuda.memory_reserved()/1e9:.1f} GB')

## 3. Zero-shot baseline

Run 5 tasks before any training to establish baseline performance.

In [ ]:
import re

# ── L-bracket geometry ────────────────────────────────────────────────────────
def _build_lbracket(plate_t: int, arm_w: int, wall_l: int, ext_l: int) -> str:
    plate_t = max(3,  min(plate_t, 30))
    arm_w   = max(10, min(arm_w,   150))
    wall_l  = max(20, min(wall_l,  300))
    ext_l   = max(20, min(ext_l,   400))
    return (
        f'import cadquery as cq\n'
        f'wall_arm = cq.Workplane("XY").box({arm_w}, {wall_l}, {plate_t})'
        f'.translate(({arm_w/2:.1f}, {wall_l/2:.1f}, 0))\n'
        f'ext_arm  = cq.Workplane("XY").box({ext_l}, {arm_w}, {plate_t})'
        f'.translate(({ext_l/2:.1f}, {arm_w/2:.1f}, 0))\n'
        f'result   = wall_arm.union(ext_arm)'
    )

# ── Prompt format ─────────────────────────────────────────────────────────────
_DIM_SYSTEM = """You are a mechanical engineer sizing an L-bracket.
Output EXACTLY 3 lines, integer mm values only. No other text.

plate_thickness: <plate Z thickness, e.g. 10>
arm_width: <width of each arm, e.g. 50>
wall_length: <length of wall arm in Y, e.g. 90>"""

def build_prompt(obs: dict) -> str:
    messages = [
        {'role': 'system', 'content': _DIM_SYSTEM},
        {'role': 'user', 'content': (
            f"{obs['task_description']}\n\n"
            f"Load: {obs['load_kg'][0]:.0f} kg | "
            f"Extension arm: {obs['extension_mm'][0]:.0f} mm | "
            f"Budget: ${obs['max_cost_usd'][0]:.0f}"
        )},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    prompt += 'plate_thickness: '
    return prompt

def _parse(text: str, key: str, default: int) -> int:
    m = re.search(rf'{key}:\s*(\d+)', text, re.IGNORECASE)
    return int(m.group(1)) if m else default

def extract_code(raw: str, extension_mm: int = 100) -> str:
    full = 'plate_thickness: ' + raw
    return _build_lbracket(
        _parse(full, 'plate_thickness', 10),
        _parse(full, 'arm_width', 50),
        _parse(full, 'wall_length', 90),
        extension_mm,
    )

# Enable Unsloth inference mode for generation (2x faster)
FastLanguageModel.for_inference(model)

@torch.no_grad()
def generate_code(obs: dict, temperature: float = 0.8) -> str:
    prompt = build_prompt(obs)
    inputs = tokenizer(prompt, return_tensors='pt',
                       truncation=True, max_length=MAX_SEQ_LEN).to(device)
    outputs = model.generate(
        **inputs, max_new_tokens=40, temperature=temperature,
        do_sample=True, pad_token_id=tokenizer.pad_token_id,
    )
    new_tokens = outputs[0][inputs['input_ids'].shape[-1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return extract_code(raw, extension_mm=int(obs['extension_mm'][0]))

# ── Baseline ───────────────────────────────────────────────────────────────────
print('Zero-shot baseline (5 tasks)')
print('-' * 50)
baseline_tasks   = env.unwrapped.task_ids()[:5]
baseline_results = []

for tid in baseline_tasks:
    obs, _ = env.reset(options={'task_id': tid})
    code = generate_code(obs)
    _, reward, _, _, info = env.step(code)
    baseline_results.append(info['success'])
    dim = info['dimension_scores']
    status = 'PASS' if info['success'] else 'FAIL'
    err = info.get('error') or (info.get('warnings') or [''])[0]
    print(f'  {tid}  {status}  reward={reward:.2f}  '
          f"S={dim.get('structural',0):.2f} C={dim.get('cost',0):.2f} "
          f"G={dim.get('geometry',0):.2f}"
          + (f'  [{err[:60]}]' if reward <= 0 else ''))

print(f'\nBaseline: {sum(baseline_results)}/5 passed')

## 4. RL Training (REINFORCE + LoRA)

VRAM during training: model (4-bit) + LoRA gradients + activations (checkpointed)
≈ 7–9 GB on T4. The 4-bit base weights never accumulate gradients — only the
LoRA adapter does, which is 40M parameters instead of 7B.

Dashboard updates every 10 episodes.

In [ ]:
N_EPISODES = 200    # Increase to 500 for meaningful improvement
VIZ_EVERY  = 10

viz = TrainingVisualizer(target_pass_rate=0.50, rolling_window=30)

# Switch model back to training mode (LoRA weights need grad)
FastLanguageModel.for_training(model)

print(f'Training {N_EPISODES} episodes | dashboard every {VIZ_EVERY}\n')
pbar = tqdm(range(1, N_EPISODES + 1), desc='Episode 1')

for episode in pbar:
    obs, info = env.reset()
    prompt    = build_prompt(obs)
    inputs    = tokenizer(prompt, return_tensors='pt',
                          truncation=True, max_length=MAX_SEQ_LEN).to(device)

    # ── Generate ──────────────────────────────────────────────────────────────
    try:
        with torch.no_grad():
            gen_out = model.generate(
                **inputs, max_new_tokens=40, temperature=0.8,
                do_sample=True, pad_token_id=tokenizer.pad_token_id,
            )
    except Exception as e:
        pbar.set_description(f'Episode {episode} — generate error')
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        optimizer.zero_grad()
        continue

    new_tokens = gen_out[0][inputs['input_ids'].shape[-1]:]
    raw        = tokenizer.decode(new_tokens, skip_special_tokens=True)
    code       = extract_code(raw, extension_mm=int(obs['extension_mm'][0]))

    obs, reward, terminated, truncated, info = env.step(code)

    # Mild safety-factor efficiency bonus (prevents blind over-building)
    sf = obs['safety_factor'][0] if obs['safety_factor'][0] > 0 else 0.0
    if sf > 0:
        sf_efficiency = min(1.0, 5.0 / max(sf, 5.0))
        reward = reward * (0.85 + 0.15 * sf_efficiency)

    pbar.set_description(f'Episode {episode}/{N_EPISODES}')
    pbar.set_postfix(
        reward=f'{reward:.2f}', sf=f'{sf:.0f}',
        S=f"{info['dimension_scores'].get('structural',0):.2f}",
        C=f"{info['dimension_scores'].get('cost',0):.2f}",
        G=f"{info['dimension_scores'].get('geometry',0):.2f}",
    )

    newly_unlocked = viz.update(episode, obs, reward, info, code)

    # ── REINFORCE update (only when reward > 0) ───────────────────────────────
    # Gradient flows only through LoRA adapter weights (~40M params).
    # Base 4-bit weights are frozen — they never accumulate gradients.
    optimizer.zero_grad()
    if reward > 0.01:
        with torch.enable_grad():
            full_ids = gen_out[0].unsqueeze(0)
            labels   = full_ids.clone()
            labels[0, :inputs['input_ids'].shape[-1]] = -100   # mask prompt
            out  = model(input_ids=full_ids, labels=labels)
            loss = out.loss * reward    # REINFORCE: scale loss by reward
            loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], 1.0
        )
        if torch.isfinite(grad_norm):
            optimizer.step()
        else:
            optimizer.zero_grad()

    # Free CUDA cache periodically (important for long runs)
    if episode % 20 == 0 and torch.cuda.is_available():
        torch.cuda.empty_cache()
        used  = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        pbar.write(f'  ep {episode}: VRAM {used:.1f}/{total:.1f} GB')

    if episode % VIZ_EVERY == 0 or newly_unlocked:
        viz.render(episode)

print('Training complete')

## 5. Final summary + champion design

In [ ]:
viz.final_summary()

## 6. Post-training evaluation on all 50 tasks

In [ ]:
# Switch to fast inference mode for evaluation
FastLanguageModel.for_inference(model)

task_ids = env.unwrapped.task_ids()
results  = {}

print('Evaluating on all 50 tasks...')
for tid in tqdm(task_ids, desc='Eval'):
    obs, _ = env.reset(options={'task_id': tid})
    code   = generate_code(obs, temperature=0.1)   # low temp = greedy
    _, reward, _, _, info = env.step(code)
    results[tid] = {
        'passed': info['success'],
        'reward': round(reward, 4),
        'dimension_scores': info['dimension_scores'],
    }

import json
from pathlib import Path
import prodata.cad_gym

tasks_path = Path(prodata.cad_gym.__file__).parent / 'tasks' / 'brackets_basic.json'
with open(tasks_path) as f:
    tasks_meta = {t['task_id']: t for t in json.load(f)}

print(f"\n{'='*60}")
print(f'Post-RL Evaluation -- {MODEL_NAME}')
print(f"{'='*60}")
for level in ('easy', 'medium', 'hard'):
    ids    = [tid for tid, t in tasks_meta.items() if t['difficulty'] == level]
    passed = sum(results[tid]['passed'] for tid in ids if tid in results)
    avg_r  = np.mean([results[tid]['reward'] for tid in ids if tid in results])
    print(f'  {level.upper():8s}: {passed}/{len(ids)} passed  avg reward {avg_r:.3f}')

total_pass = sum(r['passed'] for r in results.values())
total_avg  = np.mean([r['reward'] for r in results.values()])
print(f"{'='*60}")
print(f'  TOTAL:     {total_pass}/{len(results)} = {total_pass/len(results):.1%}  avg reward {total_avg:.3f}')
print(f'  Improvement: {total_pass/50 - sum(baseline_results)/5:+.1%} vs zero-shot')

## 7. Save LoRA adapter

In [ ]:
# Save only the LoRA adapter (~80 MB for r=16), not the full 7B model
model.save_pretrained('bracket_lora_adapter')
tokenizer.save_pretrained('bracket_lora_adapter')
print('Adapter saved to bracket_lora_adapter/')
print('Reload with: model, tokenizer = FastLanguageModel.from_pretrained("bracket_lora_adapter")')

# Optional: merge adapter into base model and save as full fp16
# model.save_pretrained_merged('bracket_merged', tokenizer, save_method='merged_16bit')

## Next steps

- **More episodes** — 500+ for meaningful improvement on 7B
- **GRPO instead of REINFORCE** — group relative policy optimization, sample 4–8 completions
  per prompt, use relative rewards. See `03_grpo_training.ipynb`.
- **Pro verifier** — `verifier_mode='pro'` to catch reward hacking. [prodata.ai](https://prodata.ai)
- **Battery gym** — try `examples/battery/03_colab_rl_training.ipynb` for the battery domain
  with pre-computed dataset (1 ms/step instead of 3 s/step)
- **Leaderboard** — [github.com/prodata-ai/ProdataGym](https://github.com/prodata-ai/ProdataGym)